# <center>Pre processing</center>
---

In [ ]:
import pandas as pd
from nltk.corpus import stopwords
import nltk
import numpy as np
import os
import sys
from nltk.corpus import stopwords
nltk.download('stopwords')
import spacy as sp
from PreProcessing.pre_processing import PreProcessing


project_root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root_path not in sys.path:
    sys.path.append(project_root_path)

In [ ]:
def remove_repetion_caracteres(string, max_repetition=2):
    if not string:
        return string
    
    result = string[0]
    count = 1
    
    for i in range(1, len(string)):
        if string[i] == string[i-1]:
            count += 1
            if count <= max_repetition:
                result += string[i]
        else:
            count = 1
            result += string[i]
    
    return result

def preprocess_text_pipeline(#input_csv_path='./data/dataFrame.csv', 
                              #output_csv_path='./data/dataFrame.csv',
                              df,
                              stopwords_file='stopwords.txt',
                              text_column="comments"):
   
    stem = sp.load("en_core_web_sm")
    pp = PreProcessing(language="en")
    
    custom_stopwords = [line.strip() for line in open(stopwords_file, 'r', encoding='utf-8')]
    english_stopwords = set(stopwords.words('english'))
    
    # Adiciona stopwords à lista da classe PreProcessing
    pp.append_stopwords_list(list(english_stopwords - set(pp.stopwords)) + custom_stopwords)

    def preprocessing(text):
        if pd.isna(text):
            return np.nan

        tokens = stem(text.lower()) # Processo de lematização da biblioteca spaCy - retorna a lista dos tokens do texto
        text = ' '.join([text for token in tokens for text in token.lemma_.strip().split()]) # Junta estes tokens na ordem do texto bruto
        text = pp.remove_stopwords(text) # Remove stopwords presentes
        text = pp.lowercase_unidecode(text) # Coloca tudo em lowercase e remove acento
        text = pp.remove_stopwords(text) # Remove stopwords presentes 
        text = pp.remove_tweet_marking(text) # Remove @ ou # seguido de 1 ou mais carcteres e n ' ' seguidos
        text = remove_repetion_caracteres(text) # Remove a repetição de caracteres ex: gooool -> gool
        text = pp.remove_urls(text) # Remove http\S+ *, ou seja, qualquer http seguido de 1 ou mais caracteres e os espaços no final
        text = pp.remove_punctuation(text) # Remove os sinais de pontuação e reorganiza os espaços
        text = pp.remove_numbers(text) # Remove os números
        text = pp.remove_n(text, n=3) # Remove palavras de tamanho <= n(n=3)
        
        return text

    df['clean_text'] = df[text_column].apply(preprocessing)
    return df

In [ ]:
df_clean = preprocess_text_pipeline(df=df, text_column='title')
display(df_clean)
df_clean.to_csv("../data/preprocessed_english_titles")